In [ ]:
pip install av

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.8/33.8 MB 1.3 MB/s eta 0:00:00


In [ ]:
import os
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_video
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.transforms import Compose, Resize, ToTensor, Normalize, Lambda
import torchvision.models as models
import torchvision.models as models
import torch.nn.functional as F
import numpy as np
import random
import av
print(av.__version__)
from tqdm import tqdm
import pandas as pd

12.0.0


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
class VideoDataset(Dataset):
    def __init__(self, root_dir, mode='train', include_mirror=True, sequence_length=30, label_file=None):
        self.root_dir = root_dir
        self.mode = mode
        self.include_mirror = include_mirror
        self.sequence_length = sequence_length
        self.feature_extractor = models.resnet18(pretrained=True)
        self.feature_extractor.eval()
        self.transform = Compose([
            Resize((224, 224)),
            Lambda(lambda x: x / 255.0),
            Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        # Load labels from CSV if available
        if label_file:
            self.label_file = os.path.join(root_dir, label_file)
            self.label_df = pd.read_csv(self.label_file)
            unique_labels = self.label_df['label'].unique()
            self.label_map = {label: idx for idx, label in enumerate(unique_labels)}
            self.filename_to_label = dict(zip(self.label_df['filename'], self.label_df['label']))

            self.reverse_label_map = {idx: label for label, idx in self.label_map.items()}

        sub_dirs = ['ALL']
        self.video_paths = []
        for sub_dir in sub_dirs:
            sub_dir_path = os.path.join(root_dir, sub_dir, mode)
            for technique in os.listdir(sub_dir_path):
                technique_path = os.path.join(sub_dir_path, technique)
                self.video_paths.extend([os.path.join(technique_path, video) for video in os.listdir(technique_path)])

    def get_string_label(self, numerical_label):
        return self.reverse_label_map.get(numerical_label, 'Unknown')

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        video, _, info = read_video(video_path, start_pts=0, end_pts=self.sequence_length, pts_unit='sec')
        video_frames = video.permute(0, 3, 1, 2)

        # Apply transformations on a per-frame basis before any concatenation
        transformed_frames = [self.transform(frame.float()) for frame in video_frames]
        video_frames = torch.stack(transformed_frames)

        if video_frames.size(0) < self.sequence_length:
            # Padding if the video is shorter than expected
            padding = torch.zeros((self.sequence_length - video_frames.size(0), 3, 224, 224))
            video_frames = torch.cat((video_frames, padding), dim=0)
        video_frames = torch.stack([self.transform(frame.float()) for frame in video_frames])

        features = self.feature_extractor(video_frames)
        features = features.view(features.size(0), -1)  # Flatten features for each frame
        if features.size(0) > self.sequence_length:
            features = features[:self.sequence_length]  # Cut off excess frames
        elif features.size(0) < self.sequence_length:
            # Pad features if there are not enough frames
            padding = torch.zeros((self.sequence_length - features.size(0), features.size(1)))
            features = torch.cat((features, padding), dim=0)
        # Retrieve label
        filename = os.path.basename(video_path)
        label_str = self.filename_to_label.get(filename, 'Unknown')
        label = self.label_map.get(label_str, -1)  # Default to -1 if no label is found

        return features, torch.tensor(label, dtype=torch.long)

# Example usage
root_dir = '/content/drive/My Drive/EC523 Final Project/FINAL DATASET'
dataset = VideoDataset(root_dir, mode='train', label_file='labels_ALL_train.csv')
dataloader = DataLoader(dataset, batch_size=30, shuffle=True)

evaluate_dataset = VideoDataset(root_dir, mode='evaluate', label_file='labels_ALL_evaluate.csv')
evaluate_loader = DataLoader(evaluate_dataset, batch_size=32, shuffle=True)


# Print DataLoader stats
print("DataLoader Train Stats:")
print("Number of batches:", len(dataloader))
print("Number of samples:", len(dataloader.dataset))

print("\nDataLoader Evaluate Stats:")
print("Number of batches:", len(evaluate_loader))
print("Number of samples:", len(evaluate_loader.dataset))


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 164MB/s]
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights

DataLoader Train Stats:
Number of batches: 61
Number of samples: 1828

DataLoader Evaluate Stats:
Number of batches: 7
Number of samples: 202


In [ ]:
class GRUCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(GRUCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size

        self.Wz = nn.Linear(input_size, hidden_size)
        self.Wr = nn.Linear(input_size, hidden_size)
        self.Wh = nn.Linear(input_size, hidden_size)

        self.Uz = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Ur = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Uh = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x, hidden):
        z = torch.sigmoid(self.Wz(x) + self.Uz(hidden))
        r = torch.sigmoid(self.Wr(x) + self.Ur(hidden))
        h_tilde = torch.tanh(self.Wh(x) + self.Uh(r * hidden))
        h_next = (1 - z) * h_tilde + z * hidden
        return h_next

In [ ]:
class GRULayer(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(GRULayer, self).__init__()
        self.hidden_size = hidden_size
        self.cell = GRUCell(input_size, hidden_size)

    def forward(self, x, hidden=None):
        if hidden is None:
            hidden = torch.zeros(x.size(0), self.hidden_size).to(x.device)

        outputs = []
        seq_len = x.size(1)
        for t in range(seq_len):
            hidden = self.cell(x[:, t, :], hidden)
            outputs.append(hidden.unsqueeze(1))
        return torch.cat(outputs, dim=1), hidden

In [ ]:
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(GRUModel, self).__init__()
        self.input_transform = nn.Linear(1000, input_size)
        self.gru = GRULayer(input_size, hidden_size)
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.input_transform(x)
        output, hidden = self.gru(x)
        logits = self.classifier(output[:, -1, :])
        return logits

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GRUModel(input_size=256, hidden_size=256, num_classes=10)
model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
def train_model(model, dataloader, criterion, optimizer, num_epochs=7, device=device):
    for epoch in tqdm(range(num_epochs), desc='Epochs', leave=True):
        model.train()
        running_loss = 0.0

        for i, (videos, labels) in tqdm(enumerate(dataloader), desc='Batches', leave=True, total=len(dataloader)):
            features = videos.to(device)
            labels = labels.to(device)  # Move labels to the device

            optimizer.zero_grad()  # Zero the gradients to prevent accumulation
            outputs = model(features)  # Forward pass
            loss = criterion(outputs, labels)  # Calculate loss
            loss.backward()  # Backpropagation
            optimizer.step()  # Update weights

            running_loss += loss.item() * features.size(0)  # Accumulate the loss

        epoch_loss = running_loss / len(dataloader.dataset)  # Calculate average loss for the epoch
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

    # Save the model after training is complete
    torch.save(model.state_dict(), '/content/drive/MyDrive/FINAL DATASET (4)/model_weights_(2).pt')
    print("Model saved successfully.")

# Example usage of the training function
train_model(model, dataloader, criterion, optimizer, num_epochs=7, device=device)

Epochs:  14%|█▍        | 1/7 [3:18:05<19:48:32, 11885.46s/it]

Epoch [1/7], Loss: 1.1053



Epochs:  29%|██▊       | 2/7 [6:26:14<16:01:12, 11534.50s/it]

Epoch [2/7], Loss: 1.0336



Epochs:  43%|████▎     | 3/7 [9:42:01<12:55:27, 11631.76s/it]

...

Epoch [5/7], Loss: 1.0318



Epochs:  86%|████████▌ | 6/7 [19:31:55<3:15:54, 11754.83s/it]

Epoch [6/7], Loss: 1.0244



Epochs: 100%|██████████| 7/7 [22:48:44<00:00, 11732.08s/it]

Epoch [7/7], Loss: 1.0149
Model saved successfully.


In [9]:
def test_model(model, dataloader, device, dataset):
    model.eval()  # Set the model to evaluation mode
    total_correct = 0
    total_samples = 0

    with torch.no_grad():  # No need to track gradients for testing
        for features, labels in tqdm(dataloader, desc='Testing', leave=False):
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features)
            _, predicted = torch.max(outputs.data, 1)  # Get the index of the max log-probability

            # Convert numerical labels back to string labels
            actual_labels = [dataset.get_string_label(label.item()) for label in labels]
            predicted_labels = [dataset.get_string_label(label.item()) for label in predicted]

            for actual, pred in zip(actual_labels, predicted_labels):
                print(f'Actual: {actual}, Predicted: {pred}')

            total_samples += labels.size(0)
            total_correct += (predicted == labels).sum().item()

    accuracy = (total_correct / total_samples) * 100
    print(f'Accuracy: {accuracy:.2f}%')

# Load model weights
model_path = '/content/drive/MyDrive/FINAL DATASET (4)/model_weights_(2).pt'
model.load_state_dict(torch.load(model_path))
model.to(device)

# Define DataLoader and Dataset for the training dataset
train_dataset = VideoDataset(root_dir, mode='train', label_file='labels_ALL_train.csv')
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Define DataLoader and Dataset for the evaluation dataset (already done in your provided code)
evaluate_dataset = VideoDataset(root_dir, mode='evaluate', label_file='labels_ALL_evaluate.csv')
evaluate_loader = DataLoader(evaluate_dataset, batch_size=32, shuffle=True)

# Test on the training dataset
print("Testing on Training Dataset:")
test_model(model, train_loader, device, train_dataset)

# Test on the evaluation dataset
print("\nTesting on Evaluation Dataset:")
test_model(model, evaluate_loader, device, evaluate_dataset)

/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are depreca

Testing on Training Dataset:


Testing:   2%|▏         | 1/58 [01:48<1:42:44, 108.15s/it]

Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Seoi nage
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Seoi nage
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Seoi nage
Actual: Osoto Gari, Predicted: Seoi nage
Actual: Seoi nage, Predicted: Seoi nage
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Uchi Mata, Predict

Testing:   3%|▎         | 2/58 [03:48<1:47:25, 115.10s/it]

...

Testing:  71%|███████▏  | 5/7 [09:59<03:54, 117.11s/it]

Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Seoi nage
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Seoi nage
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predic

Testing:  86%|████████▌ | 6/7 [12:13<02:02, 122.72s/it]

Actual: Seoi nage, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Seoi nage
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Seoi nage
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Osoto Gari
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Uchi Mata, Predicte

Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Osoto Gari
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Osoto Gari, Predicted: Seoi nage
Actual: Uchi Mata, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Actual: Seoi nage, Predicted: Uchi Mata
Accuracy: 47.52%
